# 3W Challenge Baseline: Multiclass Sample Classification

Build a sample-level classifier that predicts the operational label **0 through 9** for each labeled observation.

**Audience:** Python users who know basic pandas and classification concepts.

**Learning goals:** load fixed instance splits with 3W Toolkit, apply the
challenge label rules, train a default model five times, and summarize the
validation results. No hyperparameter search is performed.

## Prerequisites and outline

Prerequisites:

- install the repository with `pip install -e .`;
- copy `.env.example` to `.env` and configure the repository and complete
  3W Dataset v2.0.0 paths.

Outline:

1. Resolve the repository, dataset, and fixed split lists.
2. Remove unlabeled rows and map transient labels to steady fault labels.
3. Use the 3W Toolkit's fixed signal-cleaning rules.
4. Fit missing-value preprocessing independently on each training fold.
5. Fit a fresh default decision tree on each of the five fixed splits.
6. Review metrics and the aggregated confusion matrix.

In [ ]:
import os
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix
from tqdm.auto import tqdm

from ThreeWToolkit.core.enums import ModelTypeEnum
from ThreeWToolkit.dataset import ParquetDataset, ParquetDatasetConfig
from ThreeWToolkit.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
from ThreeWToolkit.models.sklearn_models import (
    SklearnModels,
    SklearnModelsConfig,
)
from ThreeWToolkit.utils.data_utils import (
    default_data_cleanup,
    get_config_dataset_ini,
)

load_dotenv()

REPOSITORY_PATH = Path(os.environ["THREE_W_REPOSITORY_PATH"]).expanduser().resolve()
DATASET_PATH = Path(os.environ["THREE_W_DATASET_PATH"]).expanduser().resolve()
CHALLENGE_PATH = Path(os.environ["THREE_W_CHALLENGE_PATH"]).expanduser().resolve()
SPLIT_DIR = Path(os.environ["THREE_W_CHALLENGE_SPLIT_DIR"]).expanduser().resolve()
RANDOM_SEED = int(os.environ["THREE_W_CHALLENGE_RANDOM_SEED"])

print(f"Repository: {REPOSITORY_PATH}")
print(f"Dataset:   {DATASET_PATH}")
print(f"Challenge: {CHALLENGE_PATH}")
print(f"Splits:    {SPLIT_DIR}")

## 1. Load every instance from the fixed splits

Each text file contains the validation instances for one run. The other four
lists form training. Their union contains all 2,228 dataset instances. Because assignment is at the instance level, samples from one Parquet file never appear in both training and validation.

In [ ]:
def read_validation_lists(split_dir: Path) -> dict[int, set[str]]:
    """Read the five one-path-per-line toolkit file lists."""
    validation_lists = {}
    for split_number in range(1, 6):
        path = split_dir / f"validation_split_{split_number}.txt"
        instances = {
            line.strip()
            for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        }
        validation_lists[split_number] = instances

    all_entries = [
        instance for instances in validation_lists.values() for instance in instances
    ]
    if len(all_entries) != len(set(all_entries)):
        raise ValueError("An instance occurs in more than one validation list.")
    return validation_lists


validation_lists = read_validation_lists(SPLIT_DIR)
instance_files = sorted(set().union(*validation_lists.values()))
if len(instance_files) != 2_228:
    raise ValueError(f"Expected all 2,228 instances, found {len(instance_files):,}.")

print(f"Using {len(instance_files):,} instance files.")
pd.Series(
    {split: len(files) for split, files in validation_lists.items()},
    name="validation_instances",
).to_frame()

## 2. Apply the challenge cleaning and label rules

The loader intentionally configures `ParquetDataset(clean_data=False)` so raw
missing labels are still visible. It then:

1. drops missing labels;
2. maps transient labels with the toolkit's configured offset (`105 → 5`);
3. keeps mapped labels `0` through `9`;
4. applies the Toolkit's fixed signal-cleaning rules.

All eligible observations are loaded. Missing-value statistics are deliberately not computed here: each validation run fits its median imputer from that run's training rows only, preventing validation instances from influencing preprocessing.

In [ ]:
DATASET_CONFIG = get_config_dataset_ini()
FEATURE_COLUMNS = [
    column
    for column in DATASET_CONFIG["COLUMNS_DATA_FILES"]
    if column not in {"timestamp", "class", "state"}
]
TRANSIENT_OFFSET = int(DATASET_CONFIG["TRANSIENT_OFFSET"])


def load_challenge_samples(
    dataset_path: Path,
    files: list[str],
    *,
    binary: bool,
) -> tuple[pd.DataFrame, int]:
    """Load, clean, and relabel every labeled observation."""
    dataset = ParquetDataset(
        ParquetDatasetConfig(
            path=dataset_path,
            split="list",
            file_list=files,
            columns=FEATURE_COLUMNS,
            target_column="class",
            clean_data=False,
            seed=RANDOM_SEED,
        )
    )

    tables = []
    unlabeled_removed = 0
    for item in tqdm(dataset.iterfiles(), total=len(dataset), unit="instance"):
        signal = item["signal"]
        raw_label = item["label"]["class"]
        valid_positions = np.flatnonzero(raw_label.notna().to_numpy())
        unlabeled_removed += len(raw_label) - len(valid_positions)
        if not len(valid_positions):
            continue

        signal = signal.iloc[valid_positions].copy()
        labels = (
            raw_label.iloc[valid_positions].astype("int64").to_numpy()
            % TRANSIENT_OFFSET
        )

        allowed = np.isin(labels, np.arange(10))
        signal = signal.iloc[np.flatnonzero(allowed)].copy()
        labels = labels[allowed]

        # Apply only fixed cleanup rules here. Fold-specific imputation happens later.
        signal = default_data_cleanup(signal)

        sample = signal.copy()
        target = labels
        if binary:
            target = (target != 0).astype("int8")
        sample["target"] = target
        sample["instance"] = item["file_name"].as_posix()
        tables.append(sample)

    if not tables:
        raise RuntimeError("No labeled samples were loaded.")
    return pd.concat(tables, ignore_index=True), unlabeled_removed

In [ ]:
samples, unlabeled_removed = load_challenge_samples(
    DATASET_PATH,
    instance_files,
    binary=False,
)
print(f"Removed {unlabeled_removed:,} unlabeled observations before cleaning.")
print(
    f"Retained {len(samples):,} observations from {samples['instance'].nunique():,} instances."
)
samples.groupby("target").size().rename("samples").to_frame()

## 3. Fit and evaluate all five splits

The model is the 3W Toolkit wrapper around scikit-learn's
`DecisionTreeClassifier`. Its model hyperparameters stay at their defaults; the
toolkit only fixes `random_state=42` for reproducibility. Before every fit, a median imputer is fitted on that split's training rows and then applied unchanged to its validation rows. Decision trees do not need feature scaling, so no global normalization is used.

`TRAINING_SAMPLE_STRIDE` controls training size without changing validation. Use `1` for every training observation or `N` for every Nth training observation. For example, `10` keeps approximately one tenth of each split's training rows. Macro-F1 and balanced accuracy give every class equal weight, making them more informative than raw accuracy for the imbalanced 3W labels.

In [ ]:
# 1 uses every training row; N uses every Nth training row.
TRAINING_SAMPLE_STRIDE = 1
if not isinstance(TRAINING_SAMPLE_STRIDE, int) or TRAINING_SAMPLE_STRIDE < 1:
    raise ValueError(
        "TRAINING_SAMPLE_STRIDE must be an integer greater than or equal to 1."
    )

print(f"Training sample stride: {TRAINING_SAMPLE_STRIDE}")


def fresh_default_model() -> SklearnModels:
    """Create a fresh toolkit decision tree with default hyperparameters."""
    return SklearnModels(
        SklearnModelsConfig(
            model_type=ModelTypeEnum.DECISION_TREE,
            random_seed=RANDOM_SEED,
        )
    )


def evaluate_five_splits(
    samples: pd.DataFrame,
    split_files: dict[int, set[str]],
    labels: list[int],
    training_sample_stride: int,
) -> tuple[pd.DataFrame, np.ndarray, dict]:
    """Fit fold-only preprocessing and a fresh model on every fixed split."""
    feature_columns = [
        column for column in samples.columns if column not in {"target", "instance"}
    ]
    available_instances = set(samples["instance"])
    rows = []
    aggregate_confusion = np.zeros((len(labels), len(labels)), dtype=int)
    effective_params = None

    for split_number in range(1, 6):
        validation_instances = split_files[split_number] & available_instances
        validation_mask = samples["instance"].isin(validation_instances)
        train_mask = ~validation_mask

        x_train = samples.loc[train_mask, feature_columns].iloc[
            ::training_sample_stride
        ]
        y_train = samples.loc[train_mask, "target"].iloc[::training_sample_stride]
        x_validation = samples.loc[validation_mask, feature_columns]
        y_validation = samples.loc[validation_mask, "target"]

        split_started = perf_counter()
        print(f"\nSplit {split_number}/5", flush=True)
        print(
            f"  Training rows: {len(x_train):,} of {int(train_mask.sum()):,} available "
            f"(stride={training_sample_stride})",
            flush=True,
        )
        print(f"  Validation rows: {len(x_validation):,}", flush=True)

        print("  Fitting the median imputer on training rows only...", flush=True)
        imputer = SimpleImputer(strategy="median")
        x_train = imputer.fit_transform(x_train)
        x_validation = imputer.transform(x_validation)

        model = fresh_default_model()
        effective_params = model.get_params()
        print("  Fitting the decision tree...", flush=True)
        model.fit(x_train, y_train)
        print("  Predicting the validation fold...", flush=True)
        prediction = model.predict(x_validation)

        row = {
            "split": split_number,
            "available_train_samples": int(train_mask.sum()),
            "train_samples": len(y_train),
            "validation_samples": int(validation_mask.sum()),
            "accuracy": accuracy_score(y_validation, prediction),
            "balanced_accuracy": balanced_accuracy_score(y_validation, prediction),
            "macro_f1": f1_score(
                y_validation,
                prediction,
                labels=labels,
                average="macro",
                zero_division=0,
            ),
        }
        rows.append(row)
        aggregate_confusion += confusion_matrix(y_validation, prediction, labels=labels)
        print(
            f"  Complete in {perf_counter() - split_started:,.1f}s: "
            f"balanced_accuracy={row['balanced_accuracy']:.4f}, "
            f"macro_f1={row['macro_f1']:.4f}",
            flush=True,
        )

    return pd.DataFrame(rows), aggregate_confusion, effective_params

In [ ]:
fold_results, aggregate_confusion, effective_params = evaluate_five_splits(
    samples,
    validation_lists,
    labels=list(range(10)),
    training_sample_stride=TRAINING_SAMPLE_STRIDE,
)
display(fold_results.round(4))
display(fold_results.mean(numeric_only=True).rename("five_split_mean").to_frame())

print("DecisionTree parameters used in every split (no search):")
effective_params

## 4. Aggregate the confusion matrices

Summing counts before row normalization gives every validation observation one
vote across the five non-overlapping validation folds.

In [ ]:
row_totals = aggregate_confusion.sum(axis=1, keepdims=True)
normalized_confusion = np.divide(
    aggregate_confusion,
    row_totals,
    out=np.zeros_like(aggregate_confusion, dtype=float),
    where=row_totals != 0,
)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    normalized_confusion,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=list(range(10)),
    yticklabels=list(range(10)),
    ax=ax,
)
ax.set(
    xlabel="Predicted class", ylabel="True class", title="Five-split confusion matrix"
)
plt.show()

## Exercise: find the hardest classes

Recall is the diagonal of the row-normalized confusion matrix. Rank the classes
to identify which event labels the default model recognizes least reliably.

In [ ]:
# Answer scaffold
recall_by_class = pd.Series(
    np.diag(normalized_confusion),
    index=list(range(10)),
    name="recall",
)
recall_by_class.sort_values().to_frame()

## Pitfalls and extensions

- **Do not fill missing labels before filtering.** The challenge rule is to
  remove them; the toolkit's default label processing uses forward/backward
  filling, so this tutorial handles labels explicitly.
- **Do not split rows randomly.** Neighboring samples are highly correlated.
  These folds keep whole instances together and also group real files from the
  same well and day.
- This default tree is a teaching baseline, not a production model.

Optional extension: keep the split files fixed and compare a window-based
feature extractor or a stricter whole-well validation design. Treat that as a
new experiment rather than tuning on these validation results.